In [1]:
# ===== 設定 =====
RUN_FROM = "vscode"          # "vscode" または "browser"
GIT_URL  = "https://github.com/kyonsy/pinns.git"
BRANCH   = "main"

In [2]:
# GPU が使えるか確認
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠ GPU が割り当てられていません。ランタイム(サーバー)を GPU で作り直してください。CPU で実行されます。")

torch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
# プロジェクトを用意して、そのフォルダに移動する
import os, glob, subprocess

if RUN_FROM == "browser":
    # Google Drive をマウントし、Drive 上のプロジェクトフォルダを使う
    from google.colab import drive
    drive.mount('/content/drive')
    # src/fcnn.py と script/poisson_eq.py があるフォルダを探す
    # 見つからない・複数ある場合は PROJECT_DIR に直接パスを書く (例: '/content/drive/MyDrive/Colab Notebooks/pinns')
    PROJECT_DIR = None
    if PROJECT_DIR is None:
        hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/**/script/poisson_eq.py', recursive=True)]
        hits = [h for h in hits if os.path.exists(os.path.join(h, 'src', 'fcnn.py'))]
        print('候補:', hits)
        assert len(hits) == 1, 'PROJECT_DIR を手で指定してください'
        PROJECT_DIR = hits[0]

elif RUN_FROM == "vscode":
    # GitHub から clone (既にあれば pull) する。サーバー上のファイルはランタイム終了で消える
    PROJECT_DIR = '/content/pinns'
    if os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
        subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only'], check=True)
    else:
        # private リポジトリなら GitHub の Personal Access Token を入力 (public なら空のまま Enter)
        from getpass import getpass
        token = getpass('GitHub token (public なら空で Enter): ').strip()
        url = GIT_URL.replace('https://', f'https://x-access-token:{token}@') if token else GIT_URL
        subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, PROJECT_DIR], check=True)
        # token を .git/config に残さない
        subprocess.run(['git', '-C', PROJECT_DIR, 'remote', 'set-url', 'origin', GIT_URL], check=True)
    print(subprocess.run(['git', '-C', PROJECT_DIR, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

else:
    raise ValueError('RUN_FROM は "vscode" か "browser"')

os.makedirs(os.path.join(PROJECT_DIR, 'out'), exist_ok=True)
os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

In [4]:
# ポアソン方程式 (結果は out/out.dat)
!python script/poisson_eq.py

In [ ]:
# 減衰振動: 学習区間と誤差の関係 (結果は out/out<j>.dat)
# !python script/dump_sys_trange.py

In [ ]:
# 減衰振動: 減衰比と誤差の関係 (結果は out/zeta<ζ>.dat)
# !python script/dump_sys_zeta.py

In [ ]:
# 結果を out.zip にまとめる (VS Code から実行したとき用)
# VS Code のエクスプローラーに Colab サーバーのファイルを表示し、/content/pinns/out.zip を右クリック→ダウンロード
# (ブラウザの Colab なら結果は Drive に直接保存されているので不要)
import shutil
print(shutil.make_archive(os.path.join(PROJECT_DIR, 'out'), 'zip', PROJECT_DIR, 'out'))